In [1]:
import sqlite3
import json

In [14]:
class ResearchSql():
    def __init__(self, db_name = 'pipeline_info.db'):
        self.conn = sqlite3.connect(db_name)
        self.conn.row_factory = sqlite3.Row # row_factory 使查询结果可以像字典一样访问
        self.cursor = self.conn.cursor()
        self._init_table()

    def _init_table(self):
        query = '''
        CREATE TABLE IF NOT EXISTS pipeline_info (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            instance_name TEXT,
            top_k_rank TEXT,
            clustered_img_path TEXT,
            status TEXT,
            error_msg TEXT,
            UNIQUE(instance_name, top_k_rank)
        )
        '''
        self.cursor.execute(query)
        self.conn.commit()

    def add_data(self, instance_name, top_k_rank, clustered_img_path, status, error_msg):
        query = '''
        INSERT INTO pipeline_info (instance_name, top_k_rank, clustered_img_path, status, error_msg)
        VALUES (?, ?, ?, ?, ?)
        ON CONFLICT(instance_name, top_k_rank) 
        DO UPDATE SET 
            clustered_img_path = excluded.clustered_img_path,
            status = excluded.status,
            error_msg = excluded.error_msg
        '''
        self.cursor.execute(query, (instance_name, top_k_rank, clustered_img_path, status, error_msg))
        self.conn.commit()

    def update_status(self, instance_name, top_k_rank, status, error_msg=None):
        query = '''
        UPDATE pipeline_info
        SET status = ?, error_msg = ?
        WHERE instance_name = ?, top_k_rank = ?
        '''
        self.cursor.execute(query, (status, error_msg, instance_name, top_k_rank))
        self.conn.commit()

    def get_instance_data(self, instance_name):
        query = '''
        SELECT *
        FROM pipeline_info
        WHERE instance_name = ?
        ORDER BY top_k_rank ASC
        '''
        self.cursor.execute(query, (instance_name,))
        rows = self.cursor.fetchall()
        result = [dict(row) for row in rows]
        return result

    def close(self):
        self.conn.close()

In [ ]:
db = ResearchSql()
# 插入数据
db.add_data('name1', '1', '/path/1', 'done', '')
db.add_data('name1', '2', '/path/2', 'running', '')
db.add_data('name1', '3', '/path/2', 'running', '')
db.add_data('name2', '2', '/path/2', 'running', '')
# 获取 JSON 数据
data = db.get_instance_data('task_01')
print('data len:', len(data))
print(data[0])
print(data[0]['instance_name'])

data len: 2
{'id': 1, 'instance_name': 'task_01', 'top_k_rank': 'rank_1', 'clustered_img_path': '/path/1', 'status': 'done', 'error_msg': ''}
task_01


In [21]:
instance_name = 'name1'
top_k_indices = []
instance_rows = db.get_instance_data(instance_name) # [{'name1':'name1, 'topk': topk..},{.}, {.}..]

for i in range(len(instance_rows)):
    top_k_indices.append(int(instance_rows[i]['top_k_rank']))
print(top_k_indices)

[1, 2, 3]
